In [34]:
import os
import sys
import json
import logging
import argparse
import torch

print("CUDA version:", torch.version.cuda)

# Show MPS device info safely
if torch.backends.mps.is_available():
    print("MPS backend is active on macOS (Metal).")
    print("Device: Apple GPU via Metal Performance Shaders (MPS).")
else:
    print("MPS not available — running on CPU instead.")

print("CPU cores:", os.cpu_count())


CUDA version: 12.8
MPS not available — running on CPU instead.
CPU cores: 64


In [35]:
import scanpy as sc

# Load your dataset
adata_sc = sc.read_h5ad("./Cell_notebooks/scrna-sciplex3/hvg.h5ad")

# Basic overview
print(adata_sc)
print("Shape:", adata_sc.shape)

# View column names (metadata about each cell)
print("Observation columns:", adata_sc.obs.columns.tolist()[:10])
print("Feature columns:", adata_sc.var_names[:10].tolist())

# How many drugs (conditions)?
print("Unique conditions:", adata_sc.obs['drug'].unique())
print("Number of cells per condition:")
print(adata_sc.obs['drug'].value_counts())



AnnData object with n_obs × n_vars = 762039 × 1000
    obs: 'size_factor', 'cell_type', 'replicate', 'dose', 'drug_code', 'pathway_level_1', 'pathway_level_2', 'product_name', 'target', 'pathway', 'drug', 'drug-dose', 'drug_code-dose', 'n_genes'
    var: 'gene_short_name', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'pca', 'rank_genes_groups'
    obsm: 'X_pca'
    varm: 'PCs', 'marker_genes-drug-rank', 'marker_genes-drug-score'
Shape: (762039, 1000)
Observation columns: ['size_factor', 'cell_type', 'replicate', 'dose', 'drug_code', 'pathway_level_1', 'pathway_level_2', 'product_name', 'target', 'pathway']
Feature columns: ['ENSG00000243620.1', 'ENSG00000271503.5', 'ENSG00000259124.1', 'ENSG00000121101.15', 'ENSG00000160963.13', 'ENSG00000135346.8', 'ENSG00000143839.14', 'ENSG00000100867.14', 'ENSG00000140986.7', 'ENSG00000230666.5']
Unique conditions: ['tak_901', 'ag_490', 'abexinostat', 'alisertib', 'busulfan', ..., 'ac480', 'cediranib', 't

In [36]:
import os
import sys
import json
import logging
import argparse
import geomloss
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List, Optional
from umap import UMAP
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import rbf_kernel
from typing import Dict, Tuple, List
from scipy.stats import ks_2samp
from scipy.spatial.distance import cdist
from sklearn.metrics import r2_score

import gc
gc.collect()

def median_heuristic_gamma(X: np.ndarray, Y: np.ndarray) -> float:
    """
    Median heuristic for RBF bandwidth: gamma = 1 / median(||x - y||^2).
    Uses the median of pairwise distances in the pooled set.
    """
    Z = np.vstack([X, Y])
    # Sample if too large for efficiency
    max_samples = 5000
    if Z.shape[0] > max_samples:
        idx = np.random.choice(Z.shape[0], size=max_samples, replace=False)
        Z = Z[idx]
    D2 = cdist(Z, Z, metric='sqeuclidean')
    # Use upper triangular without diagonal
    triu = D2[np.triu_indices_from(D2, k=1)]
    med = np.median(triu[triu > 0]) if np.any(triu > 0) else 1.0
    return 1.0 / max(med, 1e-12)

def mmd_distance(X: np.ndarray, Y: np.ndarray, gamma: float) -> float:
    """
    Unbiased MMD^2 estimator using Gaussian (RBF) kernel, sklearn backend.

    Args:
        X: (n_samples, n_features) first sample
        Y: (m_samples, n_features) second sample
        gamma: RBF kernel bandwidth; if None, uses median heuristic

    Returns:
        Unbiased MMD^2 value
    """
    n = X.shape[0]
    m = Y.shape[0]

    # Kernel matrices
    Kxx = rbf_kernel(X, X, gamma=gamma)
    Kyy = rbf_kernel(Y, Y, gamma=gamma)
    Kxy = rbf_kernel(X, Y, gamma=gamma)

    # Unbiased: exclude diagonal entries
    np.fill_diagonal(Kxx, 0.0)
    np.fill_diagonal(Kyy, 0.0)

    term_xx = Kxx.sum() / (n * (n - 1)) if n > 1 else 0.0
    term_yy = Kyy.sum() / (m * (m - 1)) if m > 1 else 0.0
    term_xy = 2.0 * Kxy.mean()

    mmd2 = term_xx + term_yy - term_xy
    mmd2 = max(mmd2, 0.0)  # Numerical stability
    return float(mmd2)

def r2_feature_means(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    R^2 computed across features between mean vectors of y_true and y_pred.
    """
    mu_true = y_true.mean(axis=0)
    mu_pred = y_pred.mean(axis=0)
    ss_res = float(np.sum((mu_true - mu_pred) ** 2))
    ss_tot = float(np.sum((mu_true - mu_true.mean()) ** 2))
    if ss_tot <= 1e-12:
        return 1.0 if ss_res <= 1e-12 else 0.0
    return 1.0 - ss_res / ss_tot

def wasserstein_pointcloud(
    X,
    Y,
    p: int = 2,
    a=None,
    b=None,
    method: str = "emd",          # "emd" (exact) or "sinkhorn" (approx)
    reg: float = 1e-1,            # Sinkhorn regularization (only used if method="sinkhorn")
    return_plan: bool = False,
):
    """
    Compute Wasserstein distance W_p between two empirical distributions supported on point sets X and Y.

    Parameters
    ----------
    X : (n, d) array-like
        Source points.
    Y : (m, d) array-like
        Target points.
    p : int
        Order of the Wasserstein distance (commonly 1 or 2).
    a : (n,) array-like or None
        Weights for X; if None, uniform weights.
    b : (m,) array-like or None
        Weights for Y; if None, uniform weights.
    method : str
        "emd" for exact optimal transport (requires POT),
        "sinkhorn" for entropic approximation (requires POT).
    reg : float
        Entropic regularization strength for Sinkhorn.
    return_plan : bool
        If True, also return the optimal transport plan.

    Returns
    -------
    Wp : float
        Wasserstein distance of order p.
    plan : (n, m) ndarray, optional
        Optimal transport plan (only if return_plan=True).
    """
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    if X.ndim != 2 or Y.ndim != 2:
        raise ValueError("X and Y must be 2D arrays with shape (n, d) and (m, d).")
    if X.shape[1] != Y.shape[1]:
        raise ValueError(f"Dimension mismatch: X has d={X.shape[1]}, Y has d={Y.shape[1]}.")

    n, d = X.shape
    m, _ = Y.shape

    if a is None:
        a = np.full(n, 1.0 / n, dtype=np.float64)
    else:
        a = np.asarray(a, dtype=np.float64)
        a = a / a.sum()

    if b is None:
        b = np.full(m, 1.0 / m, dtype=np.float64)
    else:
        b = np.asarray(b, dtype=np.float64)
        b = b / b.sum()

    # Cost matrix: C_ij = ||x_i - y_j||^p
    # Compute squared Euclidean via (x-y)^2 = x^2 + y^2 - 2xy for speed
    X2 = np.sum(X * X, axis=1, keepdims=True)          # (n, 1)
    Y2 = np.sum(Y * Y, axis=1, keepdims=True).T        # (1, m)
    sq = np.maximum(X2 + Y2 - 2.0 * (X @ Y.T), 0.0)     # (n, m)
    if p == 2:
        C = sq
    else:
        C = sq ** (p / 2.0)

    try:
        import ot  # POT: Python Optimal Transport
    except ImportError as e:
        raise ImportError(
            "This function requires the POT library. Install with: pip install pot"
        ) from e

    method = method.lower()
    if method == "emd":
        # exact OT: minimizes <P, C>
        P = ot.emd(a, b, C)
        cost = float(np.sum(P * C))
    elif method == "sinkhorn":
        # entropic OT approximation
        P = ot.sinkhorn(a, b, C, reg=reg)
        cost = float(np.sum(P * C))
    else:
        raise ValueError('method must be either "emd" or "sinkhorn".')

    Wp = cost ** (1.0 / p)

    if return_plan:
        return Wp, P
    return Wp

def summarize_metrics(y_true: np.ndarray, y_pred: np.ndarray, median_gamma: float) -> dict:
    """
    Compute a standard set of metrics: MMD^2 (RBF), R^2 of feature means, median KS across features, and Wasserstein distance.
    """
    # Drop any samples that contain NaNs in either true or pred
    mask = (~np.isnan(y_true).any(axis=1)) & (~np.isnan(y_pred).any(axis=1))
    if mask.sum() < len(y_true):
        print(f"[summarize_metrics] Dropping {len(y_true) - mask.sum()} samples with NaNs.")
    
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    out = {}

    out['mmd2_gamma_median'] = mmd_distance(y_true, y_pred, gamma=median_gamma)
    out['mmd2_gamma_0.5'] = mmd_distance(y_true, y_pred, gamma=0.5)
    out['mmd2_gamma_1.0'] = mmd_distance(y_true, y_pred, gamma=1.0)
    out['wasserstein_distance'] = wasserstein_pointcloud(y_true, y_pred, p=2, method="emd")
    out['R2_feature_means'] = r2_feature_means(y_true, y_pred)
    return out

def split_train_test(X: np.ndarray, Y: np.ndarray, train_fraction: float, seed: int = 42) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if X.shape[0] != Y.shape[0]:
        min_len = min(len(X), len(Y))
        X = X[:min_len]
        Y = Y[:min_len]

    n = X.shape[0]
    n_train = max(1, int(n * train_fraction))
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    tr_idx, te_idx = idx[:n_train], idx[n_train:]
    return X[tr_idx], X[te_idx], Y[tr_idx], Y[te_idx]

def topk_markers(adata, drug: str, k: int = 50, rank_key: str = "marker_genes-drug-rank"):
    R = adata.varm[rank_key]

    # --- get the rank vector for this drug ---
    if hasattr(R, "columns") and hasattr(R, "iloc"):  # pandas DataFrame
        if drug in R.columns:
            r = R[drug].to_numpy()
        else:
            # fallback: interpret columns as ordered groups; try to map via rank_genes_groups names
            names = adata.uns["rank_genes_groups"]["names"]
            groups = list(names.dtype.names) if (hasattr(names, "dtype") and names.dtype.names is not None) else list(names.columns)
            r = R.iloc[:, groups.index(drug)].to_numpy()
    else:  # numpy array (or array-like)
        names = adata.uns["rank_genes_groups"]["names"]
        groups = list(names.dtype.names) if (hasattr(names, "dtype") and names.dtype.names is not None) else list(names.columns)
        r = np.asarray(R)[:, groups.index(drug)]

    # smaller rank => stronger marker
    idx = np.argsort(r)[:k]
    gene_ids = adata.var_names[idx].to_list()
    gene_short = (adata.var.iloc[idx]["gene_short_name"].to_list()
                  if "gene_short_name" in adata.var.columns else None)
    return gene_ids, gene_short, idx


In [37]:
drug = "trametinib"
X_pre = adata_sc[adata_sc.obs["drug"] == "control"].copy().to_df()
X_post  = adata_sc[adata_sc.obs["drug"] == drug].copy().to_df()

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

top_genes_ids, top_genes_short, top_genes_idx = topk_markers(adata_sc, drug, k=100)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre.values, X_post.values, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)


X_pre cells: (17565, 1000)
X_post cells: (3277, 1000)
(2621, 1000)
(656, 1000)
(2621, 1000)
(656, 1000)
Median heuristic gamma: 0.002283946505187104


In [47]:
import sys
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from cellot.models.cellot import load_networks, compute_loss_f, compute_loss_g

from sklearn.metrics.pairwise import rbf_kernel


def mmd_distance(x, y, gamma):
    xx = rbf_kernel(x, x, gamma)
    xy = rbf_kernel(x, y, gamma)
    yy = rbf_kernel(y, y, gamma)

    return xx.mean() + yy.mean() - 2 * xy.mean()

def compute_mmd_loss(lhs, rhs, gammas):
    return np.mean([mmd_distance(lhs, rhs, g) for g in gammas])

from cellot.losses.mmd import mmd_distance

import sys
import numpy as np
import pandas as pd
import torch
from typing import Optional, List, Dict

from anndata import AnnData
from sklearn.preprocessing import StandardScaler

import scgen  # pip/conda package "scgen"

# -----------------------------
# scGen AE utilities
# -----------------------------
def _make_scgen_adata(X: np.ndarray, condition: str, var_names: Optional[List[str]], cond_categories: List[str]) -> AnnData:
    X = np.asarray(X, dtype=np.float32)
    if var_names is None:
        var_names = [f"g{i}" for i in range(X.shape[1])]
    obs = pd.DataFrame({
        "condition": pd.Categorical([condition] * X.shape[0], categories=cond_categories),
        "cell_type": pd.Categorical(["all"] * X.shape[0], categories=["all"]),
    })
    var = pd.DataFrame(index=pd.Index(var_names, name="gene"))
    return AnnData(X=X, obs=obs, var=var)

def train_scgen_ae(
    train_pre: np.ndarray,
    train_post: np.ndarray,
    latent_dim: int = 100,
    n_hidden: int = 512,
    n_layers: int = 2,
    early_stopping_patience: int = 100,
    dropout_rate: float = 0.0,
    max_epochs: int = 500,
    batch_size: int = 1024,
    learning_rate: float = 1e-3,
    weight_decay: float = 1e-5,
    device: str = "cuda",
    seed: int = 0,
    var_names: Optional[List[str]] = None,
    ctrl_label: str = "control",
    stim_label: str = "stim",
):
    """
    Trains an scGen model as an autoencoder on gene space (no PCA).
    Returns: model, var_names, cond_categories
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    cond_categories = [ctrl_label, stim_label]

    X_train = np.vstack([train_pre, train_post]).astype(np.float32, copy=False)
    conds = [ctrl_label] * len(train_pre) + [stim_label] * len(train_post)

    if var_names is None:
        var_names = [f"g{i}" for i in range(X_train.shape[1])]

    adata_train = AnnData(
        X=X_train,
        obs=pd.DataFrame({
            "condition": pd.Categorical(conds, categories=cond_categories),
            "cell_type": pd.Categorical(["all"] * len(conds), categories=["all"]),
        }),
        var=pd.DataFrame(index=pd.Index(var_names, name="gene")),
    )

    scgen.SCGEN.setup_anndata(adata_train, batch_key="condition", labels_key="cell_type")

    model = scgen.SCGEN(
        adata_train,
        n_hidden=n_hidden,
        n_latent=latent_dim,
        n_layers=n_layers,
        dropout_rate=dropout_rate,
    )

    use_gpu = (device.startswith("cuda") and torch.cuda.is_available())
    model.train(
        max_epochs=max_epochs,
        batch_size=batch_size,
        early_stopping=True,
        use_gpu=use_gpu,
        early_stopping_patience=int(early_stopping_patience),
        plan_kwargs={"lr": float(learning_rate), "weight_decay": float(weight_decay)},
    )
    return model, var_names, cond_categories

@torch.no_grad()
def scgen_encode(model, X: np.ndarray, condition: str, var_names, cond_categories) -> np.ndarray:
    ad = _make_scgen_adata(X, condition=condition, var_names=var_names, cond_categories=cond_categories)
    z = model.get_latent_representation(ad)
    return np.asarray(z, dtype=np.float32)

@torch.no_grad()
def scgen_decode_latent(model, Z: np.ndarray, batch_index: int = 1) -> np.ndarray:
    """
    Decode arbitrary latent Z to gene space using the model's generative path.
    batch_index=1 corresponds to the 'stim' condition if you used [control, stim] ordering.
    """
    dev = next(model.module.parameters()).device
    zt = torch.as_tensor(Z, dtype=torch.float32, device=dev)

    # scvi-tools modules typically expect batch_index as (n, 1)
    b = torch.full((zt.shape[0], 1), int(batch_index), dtype=torch.long, device=dev)

    # Some scvi modules require "library" (log-library size). If required, use zeros as a safe default.
    # (If you have raw counts + size factors, you can improve this; but this will run and is stable.)
    lib = torch.zeros((zt.shape[0], 1), dtype=torch.float32, device=dev)

    out = None
    try:
        out = model.module.generative(z=zt, batch_index=b, library=lib)
    except TypeError:
        # Older/newer signatures
        try:
            out = model.module.generative(z=zt, batch_index=b)
        except TypeError:
            out = model.module.generative(zt)

    # Pick a sensible decoded mean-like output across scvi versions
    if isinstance(out, dict):
        for key in ("px_rate", "px_scale", "px"):
            if key in out:
                x_hat = out[key]
                break
        else:
            raise KeyError(f"Could not find px_rate/px_scale/px in generative outputs: {list(out.keys())}")
    else:
        # Some implementations return tensor directly
        x_hat = out

    x_hat = x_hat.detach().cpu().numpy()
    return np.asarray(x_hat, dtype=np.float32)


# -----------------------------
#  CellOT function with scGen AE
# -----------------------------
def run_cellot_pair(
    train_pre: np.ndarray, train_post: np.ndarray,
    test_pre: np.ndarray, test_post: np.ndarray,
    layers: Optional[List[int]] = [32, 32 ,32],
    n_epochs: int = 5000,
    pca_dims: Optional[int] = None,              # kept for backward compatibility; now used as latent_dim if provided
    top_feature_subset: Optional[List[int]] = None,
    seed: int = 0,
    # scGen knobs
    scgen_max_epochs: int = 500,
    scgen_batch_size: int = 256,
    scgen_latent_dim_default: int = 50,
) -> Dict:

    device = "cuda"
    print(f"VERS torch={torch.__version__} (CellOT), device={device}", file=sys.stderr, flush=True)

    # ---- 1) Train scGen AE independently (replaces StandardScaler+PCA block) ----
    latent_dim = int(pca_dims) if (pca_dims is not None) else int(scgen_latent_dim_default)

    # Optional: if you have real gene IDs, pass them as var_names; otherwise it’s fine
    var_names = None  # or list_of_gene_ids length=1000

    scgen_model, var_names, cond_categories = train_scgen_ae(
        train_pre=train_pre,
        train_post=train_post,
        latent_dim=latent_dim,
        max_epochs=scgen_max_epochs,
        batch_size=scgen_batch_size,
        device=device,
        seed=seed,
        var_names=var_names,
        ctrl_label="control",
        stim_label="stim",
    )

    # Encode into latent space
    tr_pre_z  = scgen_encode(scgen_model, train_pre, condition="control", var_names=var_names, cond_categories=cond_categories)
    tr_post_z = scgen_encode(scgen_model, train_post, condition="stim",    var_names=var_names, cond_categories=cond_categories)
    te_pre_z  = scgen_encode(scgen_model, test_pre,  condition="control",  var_names=var_names, cond_categories=cond_categories)

    # Standardize in latent space for CellOT stability (analogous role to your PCA)
    z_scaler = StandardScaler()
    Z_all = np.vstack([tr_pre_z, tr_post_z])
    Z_all_s = z_scaler.fit_transform(Z_all)
    tr_pre_p  = Z_all_s[:len(train_pre)]
    tr_post_p = Z_all_s[len(train_pre):]
    te_pre_p  = z_scaler.transform(te_pre_z)

    # ---- 2) CellOT networks (unchanged from your code) ----
    input_dim = tr_pre_p.shape[1]
    config = {
        "model": {
            "name": "cellot",
            "hidden_units": layers,
            "kernel_init_fxn": {"name": "uniform", "a": -0.01, "b": 0.01},
            "activation": "relu",
            "softplus_W_kernels": True,
            "f": {},
            "g": {},
        }
    }
    f, g = load_networks(config, input_dim=input_dim)
    f = f.to(device).float()
    g = g.to(device).float()

    # Data tensors
    src = torch.tensor(tr_pre_p, dtype=torch.float32, device=device)
    tgt = torch.tensor(tr_post_p, dtype=torch.float32, device=device)
    te_src = torch.tensor(te_pre_p, dtype=torch.float32, device=device)

    lr = 5e-4
    optim_f = torch.optim.Adam(f.parameters(), lr=lr, betas=(0.5, 0.9), weight_decay=0)
    optim_g = torch.optim.Adam(g.parameters(), lr=lr, betas=(0.5, 0.9), weight_decay=0)

    n_epochs = n_epochs + 1
    batch_size = 256
    n_inner_iters = 10

    # ---- helper: decode a transported latent (scaled) back to gene space ----
    def _decode_from_cellot_latent(latent_scaled_np: np.ndarray) -> np.ndarray:
        # invert latent standardization
        z = z_scaler.inverse_transform(latent_scaled_np)
        # decode as "stim" condition (batch_index=1 because we used [control, stim])
        x_hat = scgen_decode_latent(scgen_model, z, batch_index=1)
        return x_hat

    # ---- 3) Training loop (same logic, but inverse-preprocess replaced by scGen decode) ----
    for epoch in range(n_epochs):
        f.train(); g.train()

        perm_t = torch.randperm(len(tgt), device=device)[:batch_size]
        yt = tgt[perm_t]

        for _ in range(n_inner_iters):
            perm_s = torch.randperm(len(src), device=device)[:batch_size]
            xs = src[perm_s].detach().clone().requires_grad_(True)

            optim_g.zero_grad()
            g_loss = compute_loss_g(f, g, xs).mean()
            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(g.parameters(), max_norm=0.5)
            optim_g.step()

        perm_s = torch.randperm(len(src), device=device)[:batch_size]
        xs = src[perm_s].detach().clone().requires_grad_(True)

        optim_f.zero_grad()
        f_loss = compute_loss_f(f, g, xs, yt).mean()
        f_loss.backward()
        optim_f.step()

        if hasattr(f, "clamp_w"):
            f.clamp_w()

        if epoch % 50 == 0:
            f.eval(); g.eval()

            # Transport training PRE (latent space), then decode to gene space
            tr_src_eval = src.detach().clone().requires_grad_(True)
            tr_pred_p = g.transport(tr_src_eval).detach().cpu().numpy()
            tr_pred = _decode_from_cellot_latent(tr_pred_p)

            train_mmd = mmd_distance(
                train_post[:, :],
                tr_pred[:, :],
                median_gamma,
            )

            te_src_full = te_src.detach().clone().requires_grad_(True)
            te_pred_full = g.transport(te_src_full).detach().cpu().numpy()
            te_pred_inv_full = _decode_from_cellot_latent(te_pred_full)

            test_mmd = mmd_distance(
                test_post[:, :],
                te_pred_inv_full[:, :],
                median_gamma,
            )
            test_mmd_cellot = compute_mmd_loss(
                test_post[:, :],
                te_pred_inv_full[:, :],
                np.logspace(1, -3, num=50),
            )

            print(
                f"[CellOT+scGen] epoch={epoch} f_loss={f_loss.item():.4f} g_loss={g_loss.item():.4f} | "
                f"train_mmd={train_mmd:.4f} | test_mmd={test_mmd:.4f} | test_mmd_cellot={test_mmd_cellot:.4f}",
                file=sys.stderr,
                flush=True,
            )

    # ---- 4) Inference ----
    f.eval(); g.eval()
    te_src_for_transport = te_src.detach().clone().requires_grad_(True)
    te_tx = g.transport(te_src_for_transport).detach().cpu().numpy()

    te_tx_inv = _decode_from_cellot_latent(te_tx)

    mmd = compute_mmd_loss(
        test_post[:len(te_tx_inv), :],
        te_tx_inv[:, :],
        gammas=np.logspace(1, -3, num=50),
    )
    print(f"[CellOT+scGen] Final CellOT MMD: {mmd:.4f}", file=sys.stderr, flush=True)

    return {"y_pred": te_tx_inv}
  


In [39]:

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 123 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = run_cellot_pair(X_tr_pre, Y_tr_post, X_te_pre, Y_te_post, n_epochs=1000, top_feature_subset=top_genes_idx, seed=seed)
  
    metrics = summarize_metrics(out["y_pred"][:, top_genes_idx], Y_te_post[:, top_genes_idx], median_gamma)
    all_metrics.append(metrics)

# Results summary
print("=== Metrics Summary over Runs for top 100 genes ===")
df = pd.DataFrame(all_metrics)
print(df.describe().T[['mean', 'std']].round(4))

VERS torch=2.8.0+cu128 (CellOT), device=cuda


(2621, 1000)
(656, 1000)
(2621, 1000)
(656, 1000)
**************** Run: 0 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 108/500:  22%|███████████████████████████████████▍                                                                                                                                | 108/500 [00:15<00:56,  6.99it/s, loss=36.2, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 250.170. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
[CellOT+scGen] epoch=0 f_loss=-7.0136 g_loss=-78.3939 | train_mmd=0.2590 | test_mmd=0.2483 | test_mmd_cellot=0.1106
[CellOT+scGen] epoch=50 f_loss=-348.3475 g_loss=3.9848 | train_mmd=0.8282 | test_mmd=0.8235 | test_mmd_cellot=0.2374
[CellOT+scGen] epoch=100 f_loss=-461.6167 g_loss=148.7183 | train_mmd=0.8258 | test_mmd=0.8211 | test_mmd_cellot=0.2360
[CellOT+scGen] epoch=150 f_loss=-0.5466 g_loss=-26.7471 | train_mmd=0.0132 | test_mmd=0.0109 | test_mmd_cellot=0.0122
[CellOT+scGen] epoch=200 f_loss=-1.8051 g_loss=-26.8757 | train_mmd=0.0092 | test_mmd=0.0067 | test_mmd_cellot=0.0086
[CellOT+scGen] epoch=250 f_loss=-1.0589 g_loss=-18.2384 | train_mmd=0.0047 | test_mmd=0.0047 | test_mmd_cellot=0.0081
[CellOT+scGen] epoch=300 f_loss=1.5194 g_loss=-18.8717 | train_

...

**************** Run: 9 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 109/500:  22%|███████████████████████████████████▊                                                                                                                                | 109/500 [00:15<00:55,  7.05it/s, loss=36.4, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 254.849. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
[CellOT+scGen] epoch=0 f_loss=-13.8845 g_loss=-73.0118 | train_mmd=0.2544 | test_mmd=0.2540 | test_mmd_cellot=0.1002
[CellOT+scGen] epoch=50 f_loss=-262.1779 g_loss=-111.0785 | train_mmd=0.8188 | test_mmd=0.8130 | test_mmd_cellot=0.2330
[CellOT+scGen] epoch=100 f_loss=-254.7059 g_loss=32.7007 | train_mmd=0.7060 | test_mmd=0.6969 | test_mmd_cellot=0.2074
[CellOT+scGen] epoch=150 f_loss=0.2071 g_loss=-23.6353 | train_mmd=0.0058 | test_mmd=0.0063 | test_mmd_cellot=0.0092
[CellOT+scGen] epoch=200 f_loss=-2.3690 g_loss=-16.4684 | train_mmd=0.0067 | test_mmd=0.0055 | test_mmd_cellot=0.0082
[CellOT+scGen] epoch=250 f_loss=0.1958 g_loss=-13.7367 | train_mmd=0.0052 | test_mmd=0.0062 | test_mmd_cellot=0.0095
[CellOT+scGen] epoch=300 f_loss=-3.9879 g_loss=-16.4666 | trai

=== Metrics Summary over Runs for top 100 genes ===
                        mean     std
mmd2_gamma_median     0.0048  0.0010
mmd2_gamma_0.5        0.0038  0.0001
mmd2_gamma_1.0        0.0033  0.0000
wasserstein_distance  5.7657  0.1089
R2_feature_means      0.7908  0.0465


In [40]:
drug = "givinostat"
X_pre = adata_sc[adata_sc.obs["drug"] == "control"].copy().to_df()
X_post  = adata_sc[adata_sc.obs["drug"] == drug].copy().to_df()

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

top_genes_ids, top_genes_short, top_genes_idx = topk_markers(adata_sc, drug, k=100)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre.values, X_post.values, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)


X_pre cells: (17565, 1000)
X_post cells: (3541, 1000)
(2832, 1000)
(709, 1000)
(2832, 1000)
(709, 1000)
Median heuristic gamma: 0.0022235993211336693


In [41]:

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = run_cellot_pair(X_tr_pre, Y_tr_post, X_te_pre, Y_te_post, n_epochs=1000, top_feature_subset=top_genes_idx, seed=seed)
  
    metrics = summarize_metrics(out["y_pred"][:, top_genes_idx], Y_te_post[:, top_genes_idx], median_gamma)
    all_metrics.append(metrics)

# Results summary
print("=== Metrics Summary over Runs for top 100 genes ===")
df = pd.DataFrame(all_metrics)
print(df.describe().T[['mean', 'std']].round(4))

VERS torch=2.8.0+cu128 (CellOT), device=cuda


(2832, 1000)
(709, 1000)
(2832, 1000)
(709, 1000)
**************** Run: 0 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 109/500:  22%|███████████████████████████████████▊                                                                                                                                | 109/500 [00:16<00:58,  6.70it/s, loss=39.3, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 261.646. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
[CellOT+scGen] epoch=0 f_loss=-4.9763 g_loss=-76.3638 | train_mmd=0.2568 | test_mmd=0.2552 | test_mmd_cellot=0.1072
[CellOT+scGen] epoch=50 f_loss=-235.0498 g_loss=-98.1081 | train_mmd=0.7659 | test_mmd=0.7656 | test_mmd_cellot=0.2056
[CellOT+scGen] epoch=100 f_loss=-343.4077 g_loss=-55.9678 | train_mmd=0.7522 | test_mmd=0.7528 | test_mmd_cellot=0.2021
[CellOT+scGen] epoch=150 f_loss=2.5484 g_loss=-31.0896 | train_mmd=0.0065 | test_mmd=0.0061 | test_mmd_cellot=0.0098
[CellOT+scGen] epoch=200 f_loss=-3.5673 g_loss=-22.2871 | train_mmd=0.0063 | test_mmd=0.0051 | test_mmd_cellot=0.0071
[CellOT+scGen] epoch=250 f_loss=2.7285 g_loss=-21.6407 | train_mmd=0.0078 | test_mmd=0.0084 | test_mmd_cellot=0.0120
[CellOT+scGen] epoch=300 f_loss=3.5614 g_loss=-18.0221 | train_

...

**************** Run: 9 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 109/500:  22%|███████████████████████████████████▊                                                                                                                                | 109/500 [00:16<00:58,  6.72it/s, loss=39.4, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 267.168. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
[CellOT+scGen] epoch=0 f_loss=-7.1265 g_loss=-84.1265 | train_mmd=0.2905 | test_mmd=0.2857 | test_mmd_cellot=0.1191
[CellOT+scGen] epoch=50 f_loss=-301.8036 g_loss=-82.5233 | train_mmd=0.7640 | test_mmd=0.7629 | test_mmd_cellot=0.2045
[CellOT+scGen] epoch=100 f_loss=-180.9032 g_loss=6.2942 | train_mmd=0.5513 | test_mmd=0.5406 | test_mmd_cellot=0.1588
[CellOT+scGen] epoch=150 f_loss=-0.6644 g_loss=-25.1194 | train_mmd=0.0045 | test_mmd=0.0053 | test_mmd_cellot=0.0094
[CellOT+scGen] epoch=200 f_loss=-0.6597 g_loss=-21.0060 | train_mmd=0.0076 | test_mmd=0.0073 | test_mmd_cellot=0.0086
[CellOT+scGen] epoch=250 f_loss=1.0323 g_loss=-18.4527 | train_mmd=0.0059 | test_mmd=0.0059 | test_mmd_cellot=0.0091
[CellOT+scGen] epoch=300 f_loss=-1.7685 g_loss=-15.4882 | train_

=== Metrics Summary over Runs for top 100 genes ===
                        mean     std
mmd2_gamma_median     0.0079  0.0011
mmd2_gamma_0.5        0.0031  0.0001
mmd2_gamma_1.0        0.0029  0.0000
wasserstein_distance  6.6746  0.0545
R2_feature_means      0.8618  0.0272


In [42]:
drug = "abexinostat"
X_pre = adata_sc[adata_sc.obs["drug"] == "control"].copy().to_df()
X_post  = adata_sc[adata_sc.obs["drug"] == drug].copy().to_df()

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

top_genes_ids, top_genes_short, top_genes_idx = topk_markers(adata_sc, drug, k=100)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre.values, X_post.values, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)


X_pre cells: (17565, 1000)
X_post cells: (4505, 1000)
(3604, 1000)
(901, 1000)
(3604, 1000)
(901, 1000)
Median heuristic gamma: 0.0021515925828798567


In [43]:

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = run_cellot_pair(X_tr_pre, Y_tr_post, X_te_pre, Y_te_post, n_epochs=1000, top_feature_subset=top_genes_idx, seed=seed)
  
    metrics = summarize_metrics(out["y_pred"][:, top_genes_idx], Y_te_post[:, top_genes_idx], median_gamma)
    all_metrics.append(metrics)

# Results summary
print("=== Metrics Summary over Runs for top 100 genes ===")
df = pd.DataFrame(all_metrics)
print(df.describe().T[['mean', 'std']].round(4))

VERS torch=2.8.0+cu128 (CellOT), device=cuda


(3604, 1000)
(901, 1000)
(3604, 1000)
(901, 1000)
**************** Run: 0 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 107/500:  21%|███████████████████████████████████                                                                                                                                 | 107/500 [00:20<01:13,  5.33it/s, loss=45.4, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 255.577. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
[CellOT+scGen] epoch=0 f_loss=-8.7540 g_loss=-69.5967 | train_mmd=0.4172 | test_mmd=0.4139 | test_mmd_cellot=0.1490
[CellOT+scGen] epoch=50 f_loss=-234.6461 g_loss=-113.7033 | train_mmd=0.7647 | test_mmd=0.7634 | test_mmd_cellot=0.1901
[CellOT+scGen] epoch=100 f_loss=-84.3129 g_loss=-0.7765 | train_mmd=0.2870 | test_mmd=0.2683 | test_mmd_cellot=0.0861
[CellOT+scGen] epoch=150 f_loss=3.9224 g_loss=-25.6733 | train_mmd=0.0060 | test_mmd=0.0068 | test_mmd_cellot=0.0090
[CellOT+scGen] epoch=200 f_loss=-0.1966 g_loss=-22.6119 | train_mmd=0.0044 | test_mmd=0.0058 | test_mmd_cellot=0.0077
[CellOT+scGen] epoch=250 f_loss=-1.4230 g_loss=-20.2134 | train_mmd=0.0053 | test_mmd=0.0063 | test_mmd_cellot=0.0087
[CellOT+scGen] epoch=300 f_loss=-0.5249 g_loss=-18.5633 | train

...

**************** Run: 9 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 107/500:  21%|███████████████████████████████████                                                                                                                                 | 107/500 [00:20<01:13,  5.33it/s, loss=44.8, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 261.008. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
[CellOT+scGen] epoch=0 f_loss=-14.6835 g_loss=-91.0990 | train_mmd=0.2296 | test_mmd=0.2437 | test_mmd_cellot=0.1132
[CellOT+scGen] epoch=50 f_loss=-352.3429 g_loss=-8.9093 | train_mmd=0.7753 | test_mmd=0.7719 | test_mmd_cellot=0.1930
[CellOT+scGen] epoch=100 f_loss=-55.2367 g_loss=-4.0660 | train_mmd=0.0607 | test_mmd=0.0537 | test_mmd_cellot=0.0265
[CellOT+scGen] epoch=150 f_loss=1.9687 g_loss=-26.3257 | train_mmd=0.0065 | test_mmd=0.0063 | test_mmd_cellot=0.0090
[CellOT+scGen] epoch=200 f_loss=2.6396 g_loss=-21.5492 | train_mmd=0.0035 | test_mmd=0.0039 | test_mmd_cellot=0.0087
[CellOT+scGen] epoch=250 f_loss=-0.4764 g_loss=-22.4128 | train_mmd=0.0043 | test_mmd=0.0049 | test_mmd_cellot=0.0092
[CellOT+scGen] epoch=300 f_loss=1.5650 g_loss=-23.2929 | train_mm

=== Metrics Summary over Runs for top 100 genes ===
                        mean     std
mmd2_gamma_median     0.0074  0.0019
mmd2_gamma_0.5        0.0022  0.0000
mmd2_gamma_1.0        0.0022  0.0000
wasserstein_distance  6.7251  0.0843
R2_feature_means      0.8385  0.0568


In [49]:
drug = "trametinib"
X_pre = adata_sc[adata_sc.obs["drug"] == "control"].copy().to_df()
X_post  = adata_sc[adata_sc.obs["drug"] == drug].copy().to_df()

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

top_genes_ids, top_genes_short, top_genes_idx = topk_markers(adata_sc, drug, k=100)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre.values, X_post.values, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)



print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = run_cellot_pair(X_tr_pre[:, top_genes_idx], Y_tr_post[:, top_genes_idx], X_te_pre[:, top_genes_idx], Y_te_post[:, top_genes_idx], n_epochs=1000, seed=seed)
  
    metrics = summarize_metrics(out["y_pred"], Y_te_post[:, top_genes_idx], median_gamma)
    all_metrics.append(metrics)

# Results summary
print("=== Metrics Summary over Runs for top 100 genes ===")
df = pd.DataFrame(all_metrics)
print(df.describe().T[['mean', 'std']].round(4))

X_pre cells: (17565, 1000)
X_post cells: (3277, 1000)
(2621, 1000)
(656, 1000)
(2621, 1000)
(656, 1000)


VERS torch=2.8.0+cu128 (CellOT), device=cuda


Median heuristic gamma: 0.0022858211277792303
(2621, 1000)
(656, 1000)
(2621, 1000)
(656, 1000)
**************** Run: 0 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 101/500:  20%|█████████████████████████████████▏                                                                                                                                  | 101/500 [00:13<00:52,  7.56it/s, loss=1.02, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 57.514. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


...

**************** Run: 9 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 101/500:  20%|█████████████████████████████████▏                                                                                                                                  | 101/500 [00:13<00:52,  7.60it/s, loss=1.03, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 58.127. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
[CellOT+scGen] epoch=0 f_loss=-13.1177 g_loss=-104.6335 | train_mmd=0.3220 | test_mmd=0.3224 | test_mmd_cellot=0.1175
[CellOT+scGen] epoch=50 f_loss=-327.9053 g_loss=-153.0841 | train_mmd=0.8280 | test_mmd=0.8203 | test_mmd_cellot=0.2363
[CellOT+scGen] epoch=100 f_loss=-280.7991 g_loss=18.1874 | train_mmd=0.7999 | test_mmd=0.8005 | test_mmd_cellot=0.2303
[CellOT+scGen] epoch=150 f_loss=6.9135 g_loss=-25.5946 | train_mmd=0.0059 | test_mmd=0.0060 | test_mmd_cellot=0.0077
[CellOT+scGen] epoch=200 f_loss=-3.7753 g_loss=-17.5112 | train_mmd=0.0057 | test_mmd=0.0057 | test_mmd_cellot=0.0065
[CellOT+scGen] epoch=250 f_loss=-1.7472 g_loss=-16.1006 | train_mmd=0.0043 | test_mmd=0.0047 | test_mmd_cellot=0.0059
[CellOT+scGen] epoch=300 f_loss=1.0765 g_loss=-15.2485 | tra

=== Metrics Summary over Runs for top 100 genes ===
                        mean     std
mmd2_gamma_median     0.0040  0.0006
mmd2_gamma_0.5        0.0033  0.0000
mmd2_gamma_1.0        0.0032  0.0000
wasserstein_distance  5.7728  0.0464
R2_feature_means      0.8059  0.0372


In [50]:
drug = "givinostat"
X_pre = adata_sc[adata_sc.obs["drug"] == "control"].copy().to_df()
X_post  = adata_sc[adata_sc.obs["drug"] == drug].copy().to_df()

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

top_genes_ids, top_genes_short, top_genes_idx = topk_markers(adata_sc, drug, k=100)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre.values, X_post.values, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)



print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = run_cellot_pair(X_tr_pre[:, top_genes_idx], Y_tr_post[:, top_genes_idx], X_te_pre[:, top_genes_idx], Y_te_post[:, top_genes_idx], n_epochs=1000, seed=seed)
  
    metrics = summarize_metrics(out["y_pred"], Y_te_post[:, top_genes_idx], median_gamma)
    all_metrics.append(metrics)

# Results summary
print("=== Metrics Summary over Runs for top 100 genes ===")
df = pd.DataFrame(all_metrics)
print(df.describe().T[['mean', 'std']].round(4))

X_pre cells: (17565, 1000)
X_post cells: (3541, 1000)
(2832, 1000)
(709, 1000)
(2832, 1000)
(709, 1000)


VERS torch=2.8.0+cu128 (CellOT), device=cuda


Median heuristic gamma: 0.002211581349119358
(2832, 1000)
(709, 1000)
(2832, 1000)
(709, 1000)
**************** Run: 0 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 101/500:  20%|█████████████████████████████████▏                                                                                                                                  | 101/500 [00:13<00:55,  7.22it/s, loss=1.01, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 68.034. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


...

**************** Run: 9 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 101/500:  20%|████████████████████████████████▉                                                                                                                                  | 101/500 [00:13<00:54,  7.27it/s, loss=0.975, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 66.415. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
[CellOT+scGen] epoch=0 f_loss=-8.1279 g_loss=-128.8386 | train_mmd=0.1444 | test_mmd=0.1339 | test_mmd_cellot=0.0751
[CellOT+scGen] epoch=50 f_loss=-176.1148 g_loss=-85.7837 | train_mmd=0.7705 | test_mmd=0.7703 | test_mmd_cellot=0.2071
[CellOT+scGen] epoch=100 f_loss=4.5425 g_loss=-19.1807 | train_mmd=0.0079 | test_mmd=0.0088 | test_mmd_cellot=0.0110
[CellOT+scGen] epoch=150 f_loss=-0.9528 g_loss=-18.1223 | train_mmd=0.0108 | test_mmd=0.0095 | test_mmd_cellot=0.0102
[CellOT+scGen] epoch=200 f_loss=2.3367 g_loss=-16.5881 | train_mmd=0.0121 | test_mmd=0.0126 | test_mmd_cellot=0.0092
[CellOT+scGen] epoch=250 f_loss=3.4714 g_loss=-15.9067 | train_mmd=0.0065 | test_mmd=0.0064 | test_mmd_cellot=0.0069
[CellOT+scGen] epoch=300 f_loss=-2.8029 g_loss=-15.8983 | train_m

=== Metrics Summary over Runs for top 100 genes ===
                        mean     std
mmd2_gamma_median     0.0057  0.0018
mmd2_gamma_0.5        0.0030  0.0000
mmd2_gamma_1.0        0.0029  0.0000
wasserstein_distance  8.2032  2.2883
R2_feature_means      0.8398  0.0545


In [51]:
drug = "abexinostat"
X_pre = adata_sc[adata_sc.obs["drug"] == "control"].copy().to_df()
X_post  = adata_sc[adata_sc.obs["drug"] == drug].copy().to_df()

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

top_genes_ids, top_genes_short, top_genes_idx = topk_markers(adata_sc, drug, k=100)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre.values, X_post.values, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)



print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = run_cellot_pair(X_tr_pre[:, top_genes_idx], Y_tr_post[:, top_genes_idx], X_te_pre[:, top_genes_idx], Y_te_post[:, top_genes_idx], n_epochs=1000, seed=seed)
  
    metrics = summarize_metrics(out["y_pred"], Y_te_post[:, top_genes_idx], median_gamma)
    all_metrics.append(metrics)

# Results summary
print("=== Metrics Summary over Runs for top 100 genes ===")
df = pd.DataFrame(all_metrics)
print(df.describe().T[['mean', 'std']].round(4))

X_pre cells: (17565, 1000)
X_post cells: (4505, 1000)
(3604, 1000)
(901, 1000)
(3604, 1000)
(901, 1000)


VERS torch=2.8.0+cu128 (CellOT), device=cuda


Median heuristic gamma: 0.0021515925828798567
(3604, 1000)
(901, 1000)
(3604, 1000)
(901, 1000)
**************** Run: 0 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 101/500:  20%|████████████████████████████████▉                                                                                                                                  | 101/500 [00:17<01:09,  5.75it/s, loss=0.948, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 61.746. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


...

**************** Run: 9 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 101/500:  20%|████████████████████████████████▉                                                                                                                                  | 101/500 [00:17<01:09,  5.77it/s, loss=0.925, v_num=1]
Monitored metric elbo_validation did not improve in the last 100 records. Best score: 62.618. Signaling Trainer to stop.
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
[CellOT+scGen] epoch=0 f_loss=-10.6377 g_loss=-81.5126 | train_mmd=0.6197 | test_mmd=0.6160 | test_mmd_cellot=0.1738
[CellOT+scGen] epoch=50 f_loss=-177.2832 g_loss=-5.6082 | train_mmd=0.7485 | test_mmd=0.7478 | test_mmd_cellot=0.1865
[CellOT+scGen] epoch=100 f_loss=3.5908 g_loss=-21.5474 | train_mmd=0.0058 | test_mmd=0.0068 | test_mmd_cellot=0.0071
[CellOT+scGen] epoch=150 f_loss=0.7907 g_loss=-19.8491 | train_mmd=0.0062 | test_mmd=0.0067 | test_mmd_cellot=0.0062
[CellOT+scGen] epoch=200 f_loss=-0.1277 g_loss=-17.8203 | train_mmd=0.0050 | test_mmd=0.0058 | test_mmd_cellot=0.0055
[CellOT+scGen] epoch=250 f_loss=0.9570 g_loss=-15.9636 | train_mmd=0.0154 | test_mmd=0.0169 | test_mmd_cellot=0.0116
[CellOT+scGen] epoch=300 f_loss=0.4004 g_loss=-17.6897 | train_mmd

=== Metrics Summary over Runs for top 100 genes ===
                        mean     std
mmd2_gamma_median     0.0059  0.0018
mmd2_gamma_0.5        0.0023  0.0000
mmd2_gamma_1.0        0.0022  0.0000
wasserstein_distance  7.3865  0.2473
R2_feature_means      0.8195  0.0700
